In [1]:
from experiment import Experiment
from database import PatchDataset
from models import EnsembleEncoder, StatisticEncoder
from metrics import EnsembleMSE
from optimizer import AdamOptimizer

ds = PatchDataset(
    input_variables=['inm_t2m'],
    target_variables=['inm_t2m'],
    modes={
        'train': {"t_min": '20240901', 't_max': '20250831', 'epoch_size': 500, 'batch_size': 4},
        'test': {"t_min": '20250901', 't_max': '20251231', 'epoch_size': 100, 'batch_size': 4},
    },
    era_scales=[
    ],
    inm_scales=[
        {'id': 'regional', 'xSize': 16, 'ySize': 16, 'tSize': 16, 'xyStep': 1, 'tStep': 1},
    ],
    target_scale={'xSize': 16, 'ySize': 16, 'tSize': 16, 'xyStep': 1, 'tStep': 1},
    fix_seed=0
)

In [5]:
import pandas as pd

loss = []
for num_stats, hidden_dim in [(1, 8), (2, 8), (3, 8), (4, 8), (5, 8), (4, 4), (4, 16), (4, 32), (4, 64), (4, 128)]:
    print(num_stats, hidden_dim)
    model = EnsembleEncoder(num_stats, hidden_dim=hidden_dim, ens_size=30)
    metric = EnsembleMSE('inm_t2m')
    optimizer = AdamOptimizer(lr=0.001)
    experiment = Experiment(f'ensemble_encoder_stat{num_stats}_hd{hidden_dim}', ds, model, metric, optimizer)
    experiment.run(50)
    df = pd.read_csv(experiment.exp_dir / 'loss_history.csv')
    df['num_stats'] = num_stats
    df['hidden_dim'] = hidden_dim
    loss.append(df)

model = StatisticEncoder(ens_size=30)
metric = EnsembleMSE('inm_t2m')
optimizer = AdamOptimizer(lr=0.001)
experiment = Experiment(f'ensemble_encoder_statistic', ds, model, metric, optimizer)
experiment.run(50)
df = pd.read_csv(experiment.exp_dir / 'loss_history.csv')
df['num_stats'] = 2
df['hidden_dim'] = 0
loss.append(df)
loss = pd.concat(loss)

1 8
2 8
3 8
4 8
5 8
4 4
4 16
4 32
4 64
4 128


In [17]:
import numpy as np

num_stats = 4
hidden_dim = 32
model = EnsembleEncoder(num_stats, hidden_dim=hidden_dim, ens_size=30)
metric = EnsembleMSE('inm_t2m')
optimizer = AdamOptimizer(lr=0.001)
experiment = Experiment(f'ensemble_encoder_stat{num_stats}_hd{hidden_dim}', ds, model, metric, optimizer)
inputs, targets = ds[0]
output = experiment.model(inputs)

a = output[0, :, 0, 0, 0]
b = targets['inm_t2m'][0, :, 0, 0, 0]
print(a.mean(), b.mean())
print(a.std(), b.std())
print(np.round(np.array(sorted(a.tolist())), 2))
print(np.round(np.array(sorted(b.tolist())), 2))

tensor(0.6678, grad_fn=<MeanBackward0>) tensor(0.6622)
tensor(0.9089, grad_fn=<StdBackward0>) tensor(0.9219)
[-1.54 -0.96 -0.75 -0.56 -0.32 -0.18 -0.01  0.08  0.18  0.27  0.36  0.48
  0.59  0.68  0.76  0.85  0.94  0.99  1.08  1.15  1.2   1.32  1.38  1.45
  1.54  1.64  1.72  1.76  1.91  2.04]
[-1.33 -0.83 -0.83 -0.68 -0.68 -0.48 -0.18 -0.13  0.47  0.57  0.57  0.57
  0.62  0.67  0.77  0.97  1.02  1.02  1.07  1.07  1.12  1.32  1.37  1.37
  1.37  1.72  1.72  1.77  1.82  2.17]


In [6]:
loss[loss['epoch'] == 50].round(2)

,epoch,train,test,num_stats,hidden_dim
49,50,1.20,1.91,1,8
49,50,0.42,0.71,2,8
49,50,0.25,0.41,3,8
49,50,0.24,0.38,4,8
49,50,0.26,0.42,5,8
49,50,0.47,0.78,4,4
49,50,0.16,0.26,4,16
49,50,0.15,0.24,4,32
49,50,0.16,0.22,4,64
49,50,0.14,0.22,4,128
